In [ ]:
# Debug Ed25519 Key Format
# This cell helps diagnose the key format issue

import os
import base64
from dotenv import load_dotenv
from pathlib import Path

# Load env
def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not find project root")

PROJECT_ROOT = find_project_root(Path.cwd())
load_dotenv(PROJECT_ROOT / ".env")

# Get the secret
api_secret = os.getenv("COINBASE_API_SECRET")
print(f"Secret (base64): {api_secret}")
print(f"Secret length: {len(api_secret)} chars")

# Decode it
secret_bytes = base64.b64decode(api_secret)
print(f"\nDecoded bytes length: {len(secret_bytes)} bytes")
print(f"First 10 bytes (hex): {secret_bytes[:10].hex()}")
print(f"Last 10 bytes (hex): {secret_bytes[-10:].hex()}")

# Ed25519 keys are 32 bytes for private key seed
# Coinbase might store seed (32) + public key (32) = 64 bytes
# Or other format
if len(secret_bytes) == 32:
    print("\n✓ This is a standard 32-byte Ed25519 private key seed")
elif len(secret_bytes) == 64:
    print("\n✓ This might be a 64-byte format (seed + public key)")
elif len(secret_bytes) == 66:
    print("\n✓ This is a 66-byte format (might have version bytes)")
else:
    print(f"\n? Unexpected length: {len(secret_bytes)} bytes")

In [ ]:
# Try loading with different approaches
from cryptography.hazmat.primitives.asymmetric import ed25519
from cryptography.hazmat.primitives import serialization

# Approach 1: First 32 bytes
print("Approach 1: Using first 32 bytes as seed...")
try:
    key1 = ed25519.Ed25519PrivateKey.from_private_bytes(secret_bytes[:32])
    print("✓ Success with first 32 bytes")
    
    # Get public key
    public_key = key1.public_key()
    public_bytes = public_key.public_bytes(
        encoding=serialization.Encoding.Raw,
        format=serialization.PublicFormat.Raw
    )
    print(f"  Public key (hex): {public_bytes.hex()}")
    print(f"  Public key (base64): {base64.b64encode(public_bytes).decode()}")
    
except Exception as e:
    print(f"✗ Failed: {e}")

# Approach 2: Check if bytes 2-34 are the seed (skip version bytes)
print("\nApproach 2: Using bytes 2-34 (skip 2-byte version)...")
try:
    key2 = ed25519.Ed25519PrivateKey.from_private_bytes(secret_bytes[2:34])
    print("✓ Success with bytes 2-34")
except Exception as e:
    print(f"✗ Failed: {e}")

# Approach 3: Try last 32 bytes
print("\nApproach 3: Using last 32 bytes...")
try:
    key3 = ed25519.Ed25519PrivateKey.from_private_bytes(secret_bytes[-32:])
    print("✓ Success with last 32 bytes")
except Exception as e:
    print(f"✗ Failed: {e}")

In [ ]:
# If 66 bytes: check structure
if len(secret_bytes) == 66:
    print("Analyzing 66-byte structure:")
    print(f"Byte 0 (version?): 0x{secret_bytes[0]:02x}")
    print(f"Byte 1 (flags?):   0x{secret_bytes[1]:02x}")
    print(f"Bytes 2-34 (32 bytes - might be private seed): {secret_bytes[2:34].hex()}")
    print(f"Bytes 34-66 (32 bytes - might be public key): {secret_bytes[34:66].hex()}")